# UltraSam on Bouchet: setup + first test

# Zhenia Rudyk
# NetID: yr245

- The first experiment used the same train CSV for train and validation, so the validation accuracy is optimistic and was only meant as a pipeline check.
- The classification task used here is:

```text
Cls-Six_2.FETAL_PLANES_ZENODO_12,400.csv
```

- Random guess for this 6-class task is about 1/6 = 0.167.
- Our first test reached about 0.669 validation accuracy after 1 epoch, which is a strong sign that the pipeline is working.

- You may need to adjust the code with your NetID

## Download Data

Download the challenge data from the challenge/Codabench site to your local machine first:
- `train.7z`
- optional task-specific files like `fetal_femur_train.zip`

## Upload Data
Upload them to:

```text
/home/yr245/project/ultrasam_project/
```

After upload, we used this structure:

```text
/home/yr245/project/ultrasam_project/
├── UltraSam/
├── train.7z
├── fetal_femur_train.zip
├── data_train7z/
└── data_fetal_femur/
```

In [ ]:
# 1) Go to the main project directory
cd ~/project
mkdir -p ultrasam_project
cd ultrasam_project
pwd
ls -lh

## 2) Get a GPU allocation

Request a GPU.

In [ ]:
salloc -p gpu_devel --gpus=1 --cpus-per-task=4 --time=02:00:00
hostname
nvidia-smi

## 3) Load miniconda and activate the existing UltraSam env

One common Bouchet issue was:
- `conda: command not found`
- or a conflict because the `Python` module was already loaded

The fix was:
1. unload the `Python` module if needed
2. load `miniconda`
3. source conda init
4. activate the env by full path

In [ ]:
module unload Python
module load miniconda
source "$(conda info --base)/etc/profile.d/conda.sh"
conda activate /home/yr245/project/ultrasam_project/.conda/envs/UltraSam

which python
python -c "import torch; print(torch.__version__, torch.cuda.is_available())"

## 4) UltraSam environment setup

If the environment does not exist yet, these are the commands we used originally.

In [ ]:
cd ~/project/ultrasam_project

module unload Python
module load miniconda
source "$(conda info --base)/etc/profile.d/conda.sh"

conda create -n UltraSam python=3.8 -y
conda activate UltraSam

git clone https://github.com/CAMMA-public/UltraSam
cd UltraSam

pip install torch==2.0.0 torchvision==0.15.1 torchaudio==2.0.1 --index-url https://download.pytorch.org/whl/cu118
pip install -U openmim
mim install mmengine
mim install "mmcv==2.1.0"
mim install mmdet
mim install mmpretrain
pip install tensorboard
pip install pandas pillow tqdm

wget -O UltraSam.pth https://s3.unistra.fr/camma_public/github/ultrasam/UltraSam.pth

## 5) CUDA / cuDNN fix that we needed

A real error we hit was:

```text
Could not load library libcudnn_cnn_infer.so.8
Error: libnvrtc.so: cannot open shared object file
```

The fix inside the active UltraSam env was:

In [ ]:
conda install -y -c nvidia cudnn=8.9.2
conda install -y -c nvidia cuda-toolkit=11.8

export LD_LIBRARY_PATH="$CONDA_PREFIX/lib:$LD_LIBRARY_PATH"
export PYTHONPATH=$PYTHONPATH:.

python -c "import torch; print(torch.cuda.is_available())"
ls $CONDA_PREFIX/lib/libcudnn_cnn_infer.so.8
ls $CONDA_PREFIX/lib/libnvrtc.so*

## 6) Extract the data

Another error we hit was that `7z` was not available by default. The fix was to load `p7zip`.

In [ ]:
cd ~/project/ultrasam_project

module load p7zip
which 7z

mkdir -p data_train7z
mkdir -p data_fetal_femur

7z x train.7z -odata_train7z
unzip -q fetal_femur_train.zip -d data_fetal_femur

## 7) Inspect the extracted train structure

In [ ]:
find data_train7z -maxdepth 2 | head -100
find data_train7z -type f | grep 'Cls'

We used this classification CSV:

```text
/home/yr245/project/ultrasam_project/data_train7z/train/csv_files/Cls-Six_2.FETAL_PLANES_ZENODO_12,400.csv
```

In [ ]:
python - <<'PY'
import pandas as pd

csv_path = "/home/yr245/project/ultrasam_project/data_train7z/train/csv_files/Cls-Six_2.FETAL_PLANES_ZENODO_12,400.csv"
df = pd.read_csv(csv_path)

print(df.head())
print(df.columns.tolist())
print("rows:", len(df))
print("task_id unique:", df["task_id"].unique())
print("labels:", sorted(df["mask"].unique()))
PY

## 8) Check that relative image paths resolve correctly

In [ ]:
python - <<'PY'
import os
import pandas as pd
from pathlib import Path

csv_path = Path("/home/yr245/project/ultrasam_project/data_train7z/train/csv_files/Cls-Six_2.FETAL_PLANES_ZENODO_12,400.csv")
df = pd.read_csv(csv_path)

print("CSV dir:", csv_path.parent)
for p in df["image_path"].head(10):
    rp = (csv_path.parent / p).resolve()
    print("raw:", p)
    print("resolved:", rp)
    print("exists:", os.path.exists(rp))
    print("-" * 60)
PY

## 9) Create the classification training script


In [ ]:
cd /home/yr245/project/ultrasam_project/UltraSam

cat > train_ultrasam_classification.py <<'PY'
import os
import json
import argparse
from pathlib import Path

import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm

from mmengine.runner.checkpoint import _load_checkpoint
from mmpretrain.models.backbones import ViTSAM


def seed_everything(seed: int = 42):
    import random
    import numpy as np
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


class UltrasoundClassificationCSVDataset(Dataset):
    def __init__(self, csv_path: str, transform=None):
        self.csv_path = Path(csv_path)
        self.csv_dir = self.csv_path.parent
        self.df = pd.read_csv(self.csv_path)
        self.transform = transform

        required_cols = {"image_path", "mask"}
        missing = required_cols - set(self.df.columns)
        if missing:
            raise ValueError(f"Missing required columns: {missing}")

        self.df["label"] = self.df["mask"].astype(int)

        resolved = []
        for p in self.df["image_path"].tolist():
            p = Path(p)
            if p.is_absolute():
                resolved.append(str(p))
            else:
                resolved.append(str((self.csv_dir / p).resolve()))
        self.df["resolved_image_path"] = resolved

        missing_files = [p for p in self.df["resolved_image_path"] if not os.path.exists(p)]
        if missing_files:
            preview = "\n".join(missing_files[:5])
            raise FileNotFoundError(
                f"{len(missing_files)} image files not found. First few:\n{preview}"
            )

        self.num_classes = int(self.df["label"].max()) + 1

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["resolved_image_path"]).convert("RGB")
        label = int(row["label"])

        if self.transform is not None:
            image = self.transform(image)

        return {
            "image": image,
            "label": torch.tensor(label, dtype=torch.long),
            "path": row["resolved_image_path"],
        }


class UltraSamClassifier(nn.Module):
    def __init__(self, checkpoint_path: str, num_classes: int, freeze_backbone: bool = True, img_size: int = 1024):
        super().__init__()

        self.backbone = ViTSAM(
            arch="base",
            img_size=img_size,
            patch_size=16,
            out_channels=256,
            use_abs_pos=True,
            use_rel_pos=True,
            window_size=14,
        )

        ckpt = _load_checkpoint(checkpoint_path, map_location="cpu")
        state_dict = ckpt.get("state_dict", ckpt)

        backbone_state = {}
        for k, v in state_dict.items():
            if k.startswith("backbone."):
                backbone_state[k[len("backbone."):]] = v

        missing, unexpected = self.backbone.load_state_dict(backbone_state, strict=False)
        print(f"Loaded backbone from {checkpoint_path}")
        print(f"Missing keys: {len(missing)}")
        print(f"Unexpected keys: {len(unexpected)}")

        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

        self.backbone.eval()
        with torch.no_grad():
            dummy = torch.zeros(1, 3, img_size, img_size)
            feat = self.backbone(dummy)
            if isinstance(feat, (list, tuple)):
                feat = feat[-1]
            if feat.ndim == 4:
                feat_dim = feat.shape[1]
            elif feat.ndim == 3:
                feat_dim = feat.shape[-1]
            else:
                raise RuntimeError(f"Unexpected feature shape: {feat.shape}")

        self.classifier = nn.Linear(feat_dim, num_classes)

    def forward(self, x):
        feat = self.backbone(x)
        if isinstance(feat, (list, tuple)):
            feat = feat[-1]

        if feat.ndim == 4:
            feat = feat.mean(dim=(2, 3))
        elif feat.ndim == 3:
            feat = feat.mean(dim=1)
        else:
            raise RuntimeError(f"Unexpected feature shape: {feat.shape}")

        logits = self.classifier(feat)
        return logits


def run_epoch(model, loader, optimizer, criterion, device, train: bool):
    model.train(train)
    total_loss = 0.0
    total_correct = 0
    total_seen = 0

    pbar = tqdm(loader, leave=False)
    for batch in pbar:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        with torch.set_grad_enabled(train):
            logits = model(images)
            loss = criterion(logits, labels)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_seen += images.size(0)

        pbar.set_postfix(
            loss=f"{total_loss / max(total_seen, 1):.4f}",
            acc=f"{total_correct / max(total_seen, 1):.4f}",
        )

    return total_loss / total_seen, total_correct / total_seen


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--train_csv", type=str, required=True)
    parser.add_argument("--val_csv", type=str, required=True)
    parser.add_argument("--checkpoint", type=str, default="UltraSam.pth")
    parser.add_argument("--output_dir", type=str, default="cls_runs/fetal_plane")
    parser.add_argument("--epochs", type=int, default=10)
    parser.add_argument("--batch_size", type=int, default=8)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--img_size", type=int, default=1024)
    parser.add_argument("--num_workers", type=int, default=4)
    parser.add_argument("--freeze_backbone", action="store_true")
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    seed_everything(args.seed)
    os.makedirs(args.output_dir, exist_ok=True)

    transform_train = transforms.Compose([
        transforms.Resize((args.img_size, args.img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[123.675/255.0, 116.28/255.0, 103.53/255.0],
            std=[58.395/255.0, 57.12/255.0, 57.375/255.0],
        ),
    ])

    transform_val = transforms.Compose([
        transforms.Resize((args.img_size, args.img_size)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[123.675/255.0, 116.28/255.0, 103.53/255.0],
            std=[58.395/255.0, 57.12/255.0, 57.375/255.0],
        ),
    ])

    train_ds = UltrasoundClassificationCSVDataset(args.train_csv, transform=transform_train)
    val_ds = UltrasoundClassificationCSVDataset(args.val_csv, transform=transform_val)

    if train_ds.num_classes != val_ds.num_classes:
        raise ValueError(
            f"Train/val class mismatch: {train_ds.num_classes} vs {val_ds.num_classes}"
        )

    num_classes = train_ds.num_classes
    print(f"Num classes: {num_classes}")
    print(f"Train size: {len(train_ds)}")
    print(f"Val size: {len(val_ds)}")

    train_loader = DataLoader(
        train_ds, batch_size=args.batch_size, shuffle=True,
        num_workers=args.num_workers, pin_memory=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=args.batch_size, shuffle=False,
        num_workers=args.num_workers, pin_memory=True
    )

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = UltraSamClassifier(
        checkpoint_path=args.checkpoint,
        num_classes=num_classes,
        freeze_backbone=args.freeze_backbone,
        img_size=args.img_size,
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=args.lr,
        weight_decay=1e-4,
    )

    best_val_acc = -1.0
    history = []

    for epoch in range(1, args.epochs + 1):
        print(f"\nEpoch {epoch}/{args.epochs}")

        train_loss, train_acc = run_epoch(
            model, train_loader, optimizer, criterion, device, train=True
        )
        val_loss, val_acc = run_epoch(
            model, val_loader, optimizer, criterion, device, train=False
        )

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
        }
        history.append(row)

        print(json.dumps(row, indent=2))

        with open(os.path.join(args.output_dir, "history.json"), "w") as f:
            json.dump(history, f, indent=2)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            ckpt = {
                "model_state_dict": model.state_dict(),
                "num_classes": num_classes,
                "best_val_acc": best_val_acc,
                "args": vars(args),
            }
            torch.save(ckpt, os.path.join(args.output_dir, "best_model.pt"))
            print(f"Saved new best model with val_acc={best_val_acc:.4f}")

    print(f"\nDone. Best val_acc = {best_val_acc:.4f}")


if __name__ == "__main__":
    main()
PY

## 10) Verify the script is not empty

In [ ]:
ls -lh train_ultrasam_classification.py
head -20 train_ultrasam_classification.py
python train_ultrasam_classification.py --help

## 11) Run the epoch-1 smoke test

This is the exact command we used. It uses the same CSV for train and validation, so this is a pipeline check, not a proper evaluation.

In [ ]:
cd /home/yr245/project/ultrasam_project/UltraSam
export LD_LIBRARY_PATH="$CONDA_PREFIX/lib:$LD_LIBRARY_PATH"
export PYTHONPATH=$PYTHONPATH:.

python train_ultrasam_classification.py --train_csv "/home/yr245/project/ultrasam_project/data_train7z/train/csv_files/Cls-Six_2.FETAL_PLANES_ZENODO_12,400.csv" --val_csv "/home/yr245/project/ultrasam_project/data_train7z/train/csv_files/Cls-Six_2.FETAL_PLANES_ZENODO_12,400.csv" --checkpoint UltraSam.pth --output_dir cls_runs/fetal_plane_smoke --epochs 1 --batch_size 2 --num_workers 2 --freeze_backbone

## 12) What result we got

For the smoke test we got roughly:
- `train_acc ≈ 0.585`
- `val_acc ≈ 0.669`

This is strong compared to random guessing for 6 classes:
- random baseline ≈ `1/6 = 0.167`

But because train and validation were the same CSV here, the validation score is optimistic and should not be reported as a final result.

## Short interpretation
- the pipeline works
- UltraSam features transfer well to this classification task
- next step is a proper train/val split and longer runs

## 13) Recommended next runs

### Frozen backbone, longer

In [ ]:
python train_ultrasam_classification.py --train_csv "/home/yr245/project/ultrasam_project/data_train7z/train/csv_files/Cls-Six_2.FETAL_PLANES_ZENODO_12,400.csv" --val_csv "/home/yr245/project/ultrasam_project/data_train7z/train/csv_files/Cls-Six_2.FETAL_PLANES_ZENODO_12,400.csv" --checkpoint UltraSam.pth --output_dir cls_runs/fetal_plane_frozen_5ep --epochs 5 --batch_size 8 --num_workers 4 --freeze_backbone

### Fine-tuning run

In [ ]:
python train_ultrasam_classification.py --train_csv "/home/yr245/project/ultrasam_project/data_train7z/train/csv_files/Cls-Six_2.FETAL_PLANES_ZENODO_12,400.csv" --val_csv "/home/yr245/project/ultrasam_project/data_train7z/train/csv_files/Cls-Six_2.FETAL_PLANES_ZENODO_12,400.csv" --checkpoint UltraSam.pth --output_dir cls_runs/fetal_plane_finetune_5ep --epochs 5 --batch_size 4 --num_workers 4 --lr 1e-4

## 14) Main issues we hit and how to fix them

### `nvidia-smi` failed
You are on a login node, not a GPU node.  
Fix: request GPU with `salloc`.

### `conda: command not found`
You need to load `miniconda` and source conda init.

### `bash: python: command not found`
The env was not activated on the GPU node.

### `libcudnn_cnn_infer.so.8` / `libnvrtc.so` missing
Install `cudnn` and `cuda-toolkit=11.8` inside the env and export `LD_LIBRARY_PATH`.

### `7z` not found
Load `p7zip`.

### `train_ultrasam_classification.py` was empty
Recreate it with the `cat <<'PY' ... PY` block above and verify with `head`.